# Notebook 1 — Data Loading & Cleaning

**Goal:** Load all Olist raw CSVs, assess data quality, clean and join into analysis-ready tables, and export to `data/processed/`.

**Business context:** We're preparing data to answer: *What percentage of customers return after their first purchase, and what factors predict retention?*

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

RAW = Path('../Olist')
PROCESSED = Path('../data/processed')
PROCESSED.mkdir(exist_ok=True)

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

## 1. Load Raw Data

In [2]:
customers = pd.read_csv(RAW / 'olist_customers_dataset.csv')
orders    = pd.read_csv(RAW / 'olist_orders_dataset.csv')
items     = pd.read_csv(RAW / 'olist_order_items_dataset.csv')
payments  = pd.read_csv(RAW / 'olist_order_payments_dataset.csv')
reviews   = pd.read_csv(RAW / 'olist_order_reviews_dataset.csv')
products  = pd.read_csv(RAW / 'olist_products_dataset.csv')
cat_trans = pd.read_csv(RAW / 'product_category_name_translation.csv')

datasets = {
    'customers': customers, 'orders': orders, 'items': items,
    'payments': payments,   'reviews': reviews, 'products': products
}

print('--- Shape & Null Summary ---')
for name, df in datasets.items():
    nulls = df.isnull().sum().sum()
    print(f'{name:12s}: {df.shape[0]:>7,} rows x {df.shape[1]:>2} cols | {nulls:>5} nulls')

--- Shape & Null Summary ---
customers   :  99,441 rows x  5 cols |     0 nulls
orders      :  99,441 rows x  8 cols |  4908 nulls
items       : 112,650 rows x  7 cols |     0 nulls
payments    : 103,886 rows x  5 cols |     0 nulls
reviews     :  99,224 rows x  7 cols | 145903 nulls
products    :  32,951 rows x  9 cols |  2448 nulls


## 2. Parse Timestamps

In [3]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

reviews['review_creation_date'] = pd.to_datetime(reviews['review_creation_date'])

print('Date range of orders:')
print(f"  From: {orders['order_purchase_timestamp'].min().date()}")
print(f"  To:   {orders['order_purchase_timestamp'].max().date()}")

Date range of orders:
  From: 2016-09-04
  To:   2018-10-17


## 3. Order Status Distribution

We keep only `delivered` orders for revenue and retention analysis. Cancelled orders are noted but excluded.

In [4]:
status_counts = orders['order_status'].value_counts()
print(status_counts)
print(f"\nDelivered: {status_counts.get('delivered', 0) / len(orders):.1%} of all orders")

orders_delivered = orders[orders['order_status'] == 'delivered'].copy()
print(f"\nDelivered orders retained: {len(orders_delivered):,}")

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Delivered: 97.0% of all orders

Delivered orders retained: 96,478


## 4. Fix the customer_unique_id Issue

`customer_id` in the orders table links to `customers`, but the same person can appear with multiple `customer_id` values.
`customer_unique_id` is the true person identifier — we always use this for retention analysis.

In [5]:
# Check how many customer_ids map to the same customer_unique_id
dup_check = customers.groupby('customer_unique_id')['customer_id'].count()
multi = dup_check[dup_check > 1]
print(f"customer_unique_ids with >1 customer_id entry: {len(multi):,}")
print(f"This is {len(multi)/len(dup_check):.1%} of unique customers — data quality note to document.")

# Build a lookup: customer_id → customer_unique_id
id_map = customers[['customer_id', 'customer_unique_id', 'customer_city', 'customer_state']].drop_duplicates()

customer_unique_ids with >1 customer_id entry: 2,997
This is 3.1% of unique customers — data quality note to document.


## 5. Build Order-Level Revenue

Some orders have multiple payment rows (split payments). Sum payment_value per order.

In [6]:
order_revenue = items.groupby('order_id').agg(
    item_count=('order_item_id', 'max'),
    product_revenue=('price', 'sum'),
    freight_revenue=('freight_value', 'sum')
).reset_index()
order_revenue['total_order_value'] = order_revenue['product_revenue'] + order_revenue['freight_revenue']

payment_totals = payments.groupby('order_id')['payment_value'].sum().reset_index()
payment_totals.rename(columns={'payment_value': 'payment_total'}, inplace=True)

print('Revenue stats per order:')
print(order_revenue['total_order_value'].describe())

Revenue stats per order:


count   98666.00
mean      160.58
std       220.47
min         9.59
25%        61.98
50%       105.29
75%       176.87
max     13664.08
Name: total_order_value, dtype: float64


## 6. Handle Outliers in Order Value

Orders above the 99th percentile are flagged — they likely represent bulk/B2B orders and could skew averages.

In [7]:
p99 = order_revenue['total_order_value'].quantile(0.99)
outliers = order_revenue[order_revenue['total_order_value'] > p99]
print(f"99th percentile order value: R${p99:,.2f}")
print(f"Outlier orders (>{p99:.0f}): {len(outliers):,} ({len(outliers)/len(order_revenue):.1%} of orders)")

# Flag but keep — we'll exclude from averages, keep for totals
order_revenue['is_outlier'] = order_revenue['total_order_value'] > p99

99th percentile order value: R$1,063.32
Outlier orders (>1063): 987 (1.0% of orders)


## 7. Get Best Review Per Order

Some orders have multiple reviews. Keep the most recent one.

In [8]:
reviews_clean = (
    reviews
    .sort_values('review_creation_date', ascending=False)
    .drop_duplicates(subset='order_id', keep='first')
    [['order_id', 'review_score', 'review_comment_message']]
)
print(f"Reviews after deduplication: {len(reviews_clean):,}")
print(reviews_clean['review_score'].value_counts().sort_index())

Reviews after deduplication: 98,673
review_score
1    11364
2     3130
3     8133
4    19044
5    57002
Name: count, dtype: int64


## 8. Build Master Orders Table

Join everything into one flat table for analysis.

In [9]:
master = (
    orders_delivered
    .merge(id_map, on='customer_id', how='left')
    .merge(order_revenue, on='order_id', how='left')
    .merge(reviews_clean, on='order_id', how='left')
)

# Add derived date columns
master['purchase_year_month'] = master['order_purchase_timestamp'].dt.to_period('M')
master['purchase_year']       = master['order_purchase_timestamp'].dt.year

# Delivery delay in days (positive = late)
master['delivery_delay_days'] = (
    master['order_delivered_customer_date'] - master['order_estimated_delivery_date']
).dt.days

print(f"Master table shape: {master.shape}")
print(master.dtypes)

Master table shape: (96478, 21)
order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
customer_unique_id                          str
customer_city                               str
customer_state                              str
item_count                                int64
product_revenue                         float64
freight_revenue                         float64
total_order_value                       float64
is_outlier                                 bool
review_score                            float64
review_comment_message                      str
purchase_year_month                   period[M]
purchase_year                             int32
delivery

## 9. Add Product Category (English)

Translate category names and join the most common product category per order.

In [10]:
products_en = products.merge(cat_trans, on='product_category_name', how='left')
products_en['category'] = products_en['product_category_name_english'].fillna(
    products_en['product_category_name']
)

# Get primary product category per order (first item)
primary_category = (
    items
    .merge(products_en[['product_id', 'category']], on='product_id', how='left')
    .sort_values('order_item_id')
    .drop_duplicates(subset='order_id', keep='first')
    [['order_id', 'category']]
)

master = master.merge(primary_category, on='order_id', how='left')
print(f"Top 10 categories:\n{master['category'].value_counts().head(10)}")

Top 10 categories:
category
bed_bath_table           9167
health_beauty            8608
sports_leisure           7491
computers_accessories    6501
furniture_decor          6213
housewares               5688
watches_gifts            5472
telephony                4076
auto                     3793
toys                     3779
Name: count, dtype: int64


## 10. Export Cleaned Data

In [11]:
master.to_csv(PROCESSED / 'master_orders.csv', index=False)

# Also export a customer summary (one row per unique customer)
customer_summary = master.groupby('customer_unique_id').agg(
    first_purchase=('order_purchase_timestamp', 'min'),
    last_purchase=('order_purchase_timestamp', 'max'),
    order_count=('order_id', 'count'),
    total_spend=('total_order_value', 'sum'),
    avg_review_score=('review_score', 'mean'),
    customer_state=('customer_state', 'first')
).reset_index()

customer_summary['is_repeat_customer'] = customer_summary['order_count'] > 1
customer_summary.to_csv(PROCESSED / 'customer_summary.csv', index=False)

print('Exports complete.')
print(f"  master_orders.csv    → {len(master):,} rows")
print(f"  customer_summary.csv → {len(customer_summary):,} rows")
print(f"\nRepeat customer rate: {customer_summary['is_repeat_customer'].mean():.1%}")

Exports complete.
  master_orders.csv    → 96,478 rows
  customer_summary.csv → 93,358 rows

Repeat customer rate: 3.0%
